In [2]:
# @title Initialization
!pip install easyocr opencv-python-headless pytesseract pillow
import cv2
import easyocr
import numpy as np
from PIL import Image
import matplotlib.pyplot as plt
import os
from datetime import datetime
from openpyxl import Workbook
import tensorflow as tf
import numpy as np
from tensorflow.keras.preprocessing import image
import ipywidgets as widgets
from IPython.display import display, clear_output
from google.colab import files
import openpyxl
from IPython.display import display, Javascript, HTML
from google.colab.output import eval_js
from base64 import b64decode
import ipywidgets as widgets
reader = easyocr.Reader(['en'])
today_date = datetime.now().strftime("%d-%m-%Y")
file_name = f"GateData_{today_date}.xlsx"
if not os.path.exists(file_name):
    wb = Workbook()
    ws = wb.active
    ws.title = "Gate Data"
    headers = ["Sl.No","Name","Vehicle Number","Vehicle Type","Reason","In-Time","Out-Time"]
    ws.append(headers)
    wb.save(file_name)
MODEL_PATH = "/content/vehicle_classifier_model.h5"
model = tf.keras.models.load_model(MODEL_PATH)
class_names = ["Ambulance","Auto-rickshaw","Bicycle","Bike","Bus","Car","Mini-truck","Tractor","Truck"]
IMG_SIZE = 128

In [3]:
# @title
def capture_image():
    js = Javascript('''
    async function takePhoto() {
        const div = document.createElement('div');
        const capture = document.createElement('button');
        capture.textContent = "📸 Capture";
        capture.style.fontSize = "20px";

        const video = document.createElement('video');
        video.style.display = 'block';
        video.style.marginTop = "10px";
        const stream = await navigator.mediaDevices.getUserMedia({video: true});

        document.body.appendChild(div);
        div.appendChild(video);
        div.appendChild(capture);

        video.srcObject = stream;
        await video.play();

        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);

        await new Promise((resolve) => capture.onclick = resolve);

        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);

        stream.getTracks().forEach(track => track.stop());
        div.remove();

        return canvas.toDataURL('image/jpeg');
    }
    takePhoto();
    ''')

    display(js)
    data = eval_js("takePhoto()")
    binary = b64decode(data.split(',')[1])

    filename = "captured_image.jpg"
    with open(filename, 'wb') as f:
        f.write(binary)

    return filename
def extract_license_with_detection(image_path):
  """Enhanced extraction with plate region detection""" # Read image
  img = cv2.imread(image_path)
  # Handle case where image cannot be loaded
  if img is None:
    return ["Error: Image not found or could not be loaded."]

  gray = cv2.cvtColor(img, cv2.COLOR_BGR2GRAY)
   # edge detection
  edges = cv2.Canny(gray, 100, 200)
  # Finding contours
  contours, _ = cv2.findContours(edges, cv2.RETR_TREE, cv2.CHAIN_APPROX_SIMPLE)

  # Filter contours (for rectangular region)
  plate_regions = []
  for cnt in contours:
    x, y, w, h = cv2.boundingRect(cnt)
    if w > 0 and h > 0:
      aspect_ratio = w / h
      if 2 < aspect_ratio < 5 and w > 50 and h > 20:
        plate_regions.append((x, y, w, h))

  results = []
  # Extract text from all detected regions
  for x, y, w, h in plate_regions:
    plate_img = img[y:y+h, x:x+w]
    if plate_img.size > 0:
      text = reader.readtext(plate_img)
      extracted = " ".join([t[1] for t in text])
      if extracted:
        results.append(extracted)

  return results if results else ["No plate detected"]

def classify_image(image_path):
    img = cv2.imread(image_path)

    if img is None:
        print("Error loading image")
        return None
    img = cv2.resize(img, (IMG_SIZE, IMG_SIZE))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    img = img / 255.0
    img = np.expand_dims(img, axis=0)
    predictions = model.predict(img)
    class_index = np.argmax(predictions[0])

    confidence = np.max(predictions[0]) * 100

    predicted_label = class_names[class_index]

    print(f"Predicted Vehicle Type: {predicted_label}")
    print(f"Confidence: {confidence:.2f}%")

    return predicted_label

In [4]:
# @title
import pandas as pd
selected_image_path = None
selected_action = None
output_area = widgets.Output()

action_selector = widgets.ToggleButtons(
    options=['IN', 'OUT'],
    description='Action:',
    button_style='info'
)

# Name & Reason
name_input = widgets.Text(description="Name:")
reason_input = widgets.Text(description="Reason:")

# Hidden File Upload Widget
file_upload = widgets.FileUpload(
    accept='image/*',
    multiple=False
)
file_upload.layout.display = 'none'

# Buttons
upload_button = widgets.Button(description="📤 Upload Image", button_style='primary')
capture_button = widgets.Button(description="📷 Capture Image", button_style='warning')
submit_button = widgets.Button(description="✅ Submit", button_style='success')

def trigger_upload(b):
    file_upload.layout.display = 'block'

upload_button.on_click(trigger_upload)

def on_file_change(change):
    global selected_image_path

    if file_upload.value:
        uploaded_file = list(file_upload.value.values())[0]
        filename = uploaded_file['metadata']['name']
        content = uploaded_file['content']

        with open(filename, 'wb') as f:
            f.write(content)

        selected_image_path = filename

        file_upload.layout.display = 'none'

        output_area.clear_output()
        with output_area:
            print("✅ Image Ready")

file_upload.observe(on_file_change, names='value')


def capture_image(b):
    global selected_image_path

    output_area.clear_output()

    cap = cv2.VideoCapture(0)
    if not cap.isOpened():
        with output_area:
            print("❌ Camera not available")
        return

    ret, frame = cap.read()
    if ret:
        filename = "captured.jpg"
        cv2.imwrite(filename, frame)
        selected_image_path = filename

        with output_area:
            print("✅ Image Captured")
    else:
        with output_area:
            print("❌ Capture Failed")

    cap.release()

capture_button.on_click(capture_image)

def handle_submit(b):
    global selected_image_path

    output_area.clear_output()

    action = action_selector.value
    name = name_input.value.strip()
    reason = reason_input.value.strip()

    if action is None:
        print("⚠ Select IN or OUT")
        return

    if selected_image_path is None:
        print("⚠ Upload or Capture Image")
        return

    vehicle_number = extract_license_with_detection(selected_image_path)

    if vehicle_number is None:
        print("❌ Could not detect vehicle number")
        return

    # ---------------- Excel Handling ----------------
    file_name = "vehicle_log.xlsx"

    if os.path.exists(file_name):
        df = pd.read_excel(file_name)
    else:
        df = pd.DataFrame(columns=[
            "Sl.No", "Name", "Vehicle Number",
            "Vehicle Type", "Reason", "In-Time", "Out-Time"
        ])

    from datetime import datetime
    current_time = datetime.now().strftime("%H:%M:%S")

    if action == "IN":
        new_entry = {
            "Sl.No": len(df) + 1,
            "Name": name,
            "Vehicle Number": vehicle_number,
            "Vehicle Type": "Car",
            "Reason": reason,
            "In-Time": current_time,
            "Out-Time": ""
        }
        df = pd.concat([df, pd.DataFrame([new_entry])], ignore_index=True)
        print("✅ Entry Recorded Successfully")

    elif action == "OUT":
        if vehicle_number in df["Vehicle Number"].values:
            df.loc[df["Vehicle Number"] == vehicle_number, "Out-Time"] = current_time
            print("✅ Exit Recorded Successfully")
        else:
            print("⚠ Vehicle Not Registered")

    df.to_excel(file_name, index=False)

    # ---------------- CLEANUP ----------------
    if os.path.exists(selected_image_path):
        os.remove(selected_image_path)

    selected_image_path = None
    name_input.value = ""
    reason_input.value = ""
    action_selector.value = None

submit_button.on_click(handle_submit)

# ---------------- DASHBOARD ----------------
dashboard = widgets.VBox([
    widgets.HTML("<h2>🚗 SMART GATE SYSTEM</h2>"),
    action_selector,
    name_input,
    reason_input,
    widgets.HBox([upload_button, capture_button]),
    file_upload,
    submit_button,
    output_area
])

display(dashboard)

/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


✅ Entry Recorded Successfully
